# 05. Multi-Timeframe Strategy Analysis — Does Candle Size Matter?

### Key Question: Does the candle timeframe (5m, 15m, 1h, 4h) influence strategy profitability?

This notebook answers that by running **all 20 strategies** across **4 different candle timeframes** and comparing results side-by-side.

We also include **gold-backed tokens (XAUTUSDT, PAXGUSDT)** to compare crypto TA strategies against a traditional safe-haven asset.

### Timeframes Tested
| Candle | Hold Period | Data Span (~) |
|--------|------------|---------------|
| 5 min  | 1h (16 candles) | ~1.7 days |
| 15 min | 3h (16 candles) | ~5.2 days |
| 1 hour | 6h (6 candles) | ~20.8 days |
| 4 hour | 24h (6 candles) | ~83.3 days |

> **Note**: `XAUUSDT` does not exist on Binance. The gold-pegged tokens are **XAUTUSDT** (Tether Gold) and **PAXGUSDT** (PAX Gold), both tracking the price of 1 troy ounce of gold (~$4,050).

In [16]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from backend.binance_client import binance_client
from backend.indicators import enrich_klines_dataframe
import time

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 220)
print('All modules loaded.')

All modules loaded.


## 1. Configuration & Data Fetch

We pull 500 candles per symbol per timeframe. Each timeframe uses a proportional hold period (16 candles for 5m/15m, 6 candles for 1h/4h) so the *number of candles held* stays comparable.

In [17]:
# Crypto pairs + Gold pairs
CRYPTO_SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'XRPUSDT', 'ADAUSDT', 'DOGEUSDT', 'AVAXUSDT']
GOLD_SYMBOLS = ['XAUTUSDT', 'PAXGUSDT']
ALL_SYMBOLS = CRYPTO_SYMBOLS + GOLD_SYMBOLS

# Timeframe configs: (interval, hold_candles, label, hold_duration_label)
TIMEFRAMES = [
    ('5m',  12, '5-Min Candle',  '1h hold'),
    ('15m', 12, '15-Min Candle', '3h hold'),
    ('1h',   6, '1-Hour Candle', '6h hold'),
    ('4h',   6, '4-Hour Candle', '24h hold'),
]

LIMIT = 500

# Fetch all data
# Structure: datasets[interval][symbol] = DataFrame
datasets = {}
for interval, hold, label, hold_label in TIMEFRAMES:
    datasets[interval] = {}
    print(f'\n--- Fetching {label} ({interval}) data ---')
    for sym in ALL_SYMBOLS:
        try:
            raw = binance_client.get_klines(sym, interval=interval, limit=LIMIT)
            df = enrich_klines_dataframe(raw)
            if not df.empty and len(df) >= 60:
                datasets[interval][sym] = df
                asset_type = 'GOLD' if sym in GOLD_SYMBOLS else 'CRYPTO'
                print(f'  {sym} [{asset_type}]: {len(df)} candles')
            else:
                print(f'  {sym}: Insufficient data, skipped')
        except Exception as e:
            print(f'  {sym}: Error - {e}')
        time.sleep(0.1)  # rate limit courtesy

print(f'\nData fetch complete. Timeframes loaded: {len(datasets)}')


--- Fetching 5-Min Candle (5m) data ---
  BTCUSDT [CRYPTO]: 500 candles
  ETHUSDT [CRYPTO]: 500 candles
  SOLUSDT [CRYPTO]: 500 candles
  BNBUSDT [CRYPTO]: 500 candles
  XRPUSDT [CRYPTO]: 500 candles
  ADAUSDT [CRYPTO]: 500 candles
  DOGEUSDT [CRYPTO]: 500 candles
  AVAXUSDT [CRYPTO]: 500 candles
  XAUTUSDT [GOLD]: 500 candles
  PAXGUSDT [GOLD]: 500 candles

--- Fetching 15-Min Candle (15m) data ---
  BTCUSDT [CRYPTO]: 500 candles
  ETHUSDT [CRYPTO]: 500 candles
  SOLUSDT [CRYPTO]: 500 candles
  BNBUSDT [CRYPTO]: 500 candles
  XRPUSDT [CRYPTO]: 500 candles
  ADAUSDT [CRYPTO]: 500 candles
  DOGEUSDT [CRYPTO]: 500 candles
  AVAXUSDT [CRYPTO]: 500 candles
  XAUTUSDT [GOLD]: 500 candles
  PAXGUSDT [GOLD]: 500 candles

--- Fetching 1-Hour Candle (1h) data ---
  BTCUSDT [CRYPTO]: 500 candles
  ETHUSDT [CRYPTO]: 500 candles
  SOLUSDT [CRYPTO]: 500 candles
  BNBUSDT [CRYPTO]: 500 candles
  XRPUSDT [CRYPTO]: 500 candles
  ADAUSDT [CRYPTO]: 500 candles
  DOGEUSDT [CRYPTO]: 500 candles
  AVAXUSD

## 2. Strategy Definitions
Same 20 strategies from notebook 04 — now tested across every candle timeframe.

In [18]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np

def strat_rsi_oversold(df):
    """Buy when RSI drops below 30"""
    return (df['rsi'] < 30) & (df['rsi'].shift(1) >= 30)

def strat_rsi_overbought(df):
    """Short when RSI rises above 70"""
    return (df['rsi'] > 70) & (df['rsi'].shift(1) <= 70)

def strat_macd_bull_cross(df):
    """Buy on MACD histogram flip positive"""
    return (df['macd_hist'].shift(1) <= 0) & (df['macd_hist'] > 0)

def strat_macd_bear_cross(df):
    """Sell on MACD histogram flip negative"""
    return (df['macd_hist'].shift(1) >= 0) & (df['macd_hist'] < 0)

def strat_ema_golden_cross(df):
    """Buy when EMA 9 crosses above EMA 21"""
    return (df['ema_9'].shift(1) <= df['ema_21'].shift(1)) & (df['ema_9'] > df['ema_21'])

def strat_ema_death_cross(df):
    """Sell when EMA 9 crosses below EMA 21"""
    return (df['ema_9'].shift(1) >= df['ema_21'].shift(1)) & (df['ema_9'] < df['ema_21'])

def strat_bb_lower_bounce(df):
    """Buy when close dips below lower Bollinger Band"""
    return (df['close'] < df['bb_lower']) & (df['close'].shift(1) >= df['bb_lower'].shift(1))

def strat_bb_upper_reject(df):
    """Sell when close exceeds upper Bollinger Band"""
    return (df['close'] > df['bb_upper']) & (df['close'].shift(1) <= df['bb_upper'].shift(1))

def strat_volume_rsi_confirm(df):
    """Buy when RVOL > 2x AND RSI < 45"""
    return (df['rvol'] > 2.0) & (df['rsi'] < 45)

def strat_triple_ema_stack(df):
    """Buy when EMA 9 > EMA 21 > EMA 50 alignment first forms"""
    aligned = (df['ema_9'] > df['ema_21']) & (df['ema_21'] > df['ema_50'])
    prev = (df['ema_9'].shift(1) > df['ema_21'].shift(1)) & (df['ema_21'].shift(1) > df['ema_50'].shift(1))
    return aligned & (~prev)

def strat_mean_reversion(df):
    """Buy when RSI < 35 AND close below lower BB"""
    return (df['rsi'] < 35) & (df['close'] < df['bb_lower'])

def strat_momentum_breakout(df):
    """Buy on MACD bull cross + RSI > 55 + RVOL > 1.5"""
    macd_bull = (df['macd_hist'].shift(1) <= 0) & (df['macd_hist'] > 0)
    return macd_bull & (df['rsi'] > 55) & (df['rvol'] > 1.5)

def strat_stoch_rsi_oversold(df):
    """Buy when StochRSI %K crosses above %D in oversold territory (<20)"""
    return (df['stoch_k'].shift(1) <= df['stoch_d'].shift(1)) & (df['stoch_k'] > df['stoch_d']) & (df['stoch_k'] < 20)

def strat_stoch_rsi_overbought(df):
    """Sell when StochRSI %K crosses below %D in overbought territory (>80)"""
    return (df['stoch_k'].shift(1) >= df['stoch_d'].shift(1)) & (df['stoch_k'] < df['stoch_d']) & (df['stoch_k'] > 80)

def strat_bb_squeeze_breakout(df):
    """Buy when BB goes inside Keltner Channels (Squeeze) and breaks out of upper BB"""
    squeeze_on = (df['bb_upper'] < df['kc_upper']) & (df['bb_lower'] > df['kc_lower'])
    breakout = (df['close'] > df['bb_upper'])
    return squeeze_on & breakout

def strat_ema_pullback(df):
    """Buy when price is above EMA 200 (uptrend) but pulls back and crosses above EMA 50"""
    uptrend = df['close'] > df['ema_200']
    pullback_bounce = (df['close'].shift(1) < df['ema_50'].shift(1)) & (df['close'] > df['ema_50'])
    return uptrend & pullback_bounce

# --- NEW 15M INTRADAY CRYPTO STRATEGIES ---

def strat_supertrend_bullish(df):
    """Buy when Supertrend flips from Bearish (-1) to Bullish (+1)"""
    return (df['supertrend_dir'].shift(1) == -1) & (df['supertrend_dir'] == 1)

def strat_supertrend_bearish(df):
    """Short when Supertrend flips from Bullish (+1) to Bearish (-1)"""
    return (df['supertrend_dir'].shift(1) == 1) & (df['supertrend_dir'] == -1)

def strat_crypto_fib_stack(df):
    """Buy when fast crypto EMA 8 > EMA 21 > EMA 55 stack first forms"""
    aligned = (df['ema_8'] > df['ema_21']) & (df['ema_21'] > df['ema_55'])
    prev = (df['ema_8'].shift(1) > df['ema_21'].shift(1)) & (df['ema_21'].shift(1) > df['ema_55'].shift(1))
    return aligned & (~prev)

def strat_vwma_cross(df):
    """Buy when Volume-Weighted Moving Average (VWMA 8) crosses above VWMA 21"""
    return (df['vwma_8'].shift(1) <= df['vwma_21'].shift(1)) & (df['vwma_8'] > df['vwma_21'])

STRATEGIES = {
    'Supertrend Bullish (10,3)':   {'fn': strat_supertrend_bullish,   'direction': 'LONG'},
    'Supertrend Bearish (10,3)':   {'fn': strat_supertrend_bearish,   'direction': 'SHORT'},
    'Crypto Fib Stack (8/21/55)':  {'fn': strat_crypto_fib_stack,     'direction': 'LONG'},
    'VWMA Cross (8/21 Volume)':    {'fn': strat_vwma_cross,           'direction': 'LONG'},
    'StochRSI Oversold':           {'fn': strat_stoch_rsi_oversold,   'direction': 'LONG'},
    'StochRSI Overbought':         {'fn': strat_stoch_rsi_overbought, 'direction': 'SHORT'},
    'BB Squeeze Breakout':         {'fn': strat_bb_squeeze_breakout,  'direction': 'LONG'},
    'EMA 200/50 Pullback':         {'fn': strat_ema_pullback,         'direction': 'LONG'},
    'RSI Oversold Bounce':         {'fn': strat_rsi_oversold,          'direction': 'LONG'},
    'RSI Overbought Short':        {'fn': strat_rsi_overbought,        'direction': 'SHORT'},
    'Bullish MACD Cross':          {'fn': strat_macd_bull_cross,       'direction': 'LONG'},
    'Bearish MACD Cross':          {'fn': strat_macd_bear_cross,       'direction': 'SHORT'},
    'EMA 9/21 Golden Cross':       {'fn': strat_ema_golden_cross,      'direction': 'LONG'},
    'EMA 9/21 Death Cross':        {'fn': strat_ema_death_cross,       'direction': 'SHORT'},
    'BB Lower Bounce':             {'fn': strat_bb_lower_bounce,       'direction': 'LONG'},
    'BB Upper Reject':             {'fn': strat_bb_upper_reject,       'direction': 'SHORT'},
    'Volume Surge + RSI Dip':      {'fn': strat_volume_rsi_confirm,    'direction': 'LONG'},
    'Triple EMA Stack':            {'fn': strat_triple_ema_stack,      'direction': 'LONG'},
    'Mean Reversion':              {'fn': strat_mean_reversion,        'direction': 'LONG'},
    'Momentum Breakout':           {'fn': strat_momentum_breakout,     'direction': 'LONG'},
}

print(f'{len(STRATEGIES)} strategies defined (including 15m intraday Supertrend & VWMA).')


20 strategies defined (including 15m intraday Supertrend & VWMA).


## 3. Multi-Timeframe Backtesting Engine

In [19]:
def compute_trade_returns(df, signal_mask, direction, hold_period):
    """Return a Series of per-trade % returns."""
    df = df.copy()
    df['future_close'] = df['close'].shift(-hold_period)
    if direction == 'LONG':
        df['trade_return'] = ((df['future_close'] - df['close']) / df['close']) * 100
    else:
        df['trade_return'] = ((df['close'] - df['future_close']) / df['close']) * 100
    return df[signal_mask].dropna(subset=['trade_return'])['trade_return']

def calc_metrics(returns):
    """Compute performance metrics from a Series of trade returns."""
    if len(returns) == 0:
        return None
    wins = returns[returns > 0]
    losses = returns[returns <= 0]
    cum = (1 + returns / 100).cumprod()
    dd = ((cum - cum.cummax()) / cum.cummax()) * 100
    gp = wins.sum() if len(wins) > 0 else 0
    gl = abs(losses.sum()) if len(losses) > 0 else 1e-10
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0
    return {
        'Signals': len(returns),
        'Win Rate (%)': round((len(wins)/len(returns))*100, 2),
        'Avg Return (%)': round(returns.mean(), 4),
        'Median Return (%)': round(returns.median(), 4),
        'Total Return (%)': round(((1+returns/100).prod()-1)*100, 4),
        'Max Drawdown (%)': round(dd.min(), 4),
        'Profit Factor': round(gp/gl, 4),
        'Sharpe Ratio': round(sharpe, 4),
        'Best Trade (%)': round(returns.max(), 4),
        'Worst Trade (%)': round(returns.min(), 4),
    }

print('Engine ready.')

Engine ready.


In [20]:
# Run all strategies × all timeframes × all symbols
all_results = []

for interval, hold_candles, tf_label, hold_label in TIMEFRAMES:
    print(f'\nBacktesting on {tf_label} ({hold_label})...')
    sym_data = datasets.get(interval, {})
    
    for strat_name, strat_info in STRATEGIES.items():
        fn = strat_info['fn']
        direction = strat_info['direction']
        
        # --- Aggregate across CRYPTO symbols only ---
        crypto_returns = []
        for sym in CRYPTO_SYMBOLS:
            df = sym_data.get(sym)
            if df is None:
                continue
            try:
                signals = fn(df)
                if signals.sum() > 0:
                    rets = compute_trade_returns(df, signals, direction, hold_candles)
                    crypto_returns.extend(rets.tolist())
            except Exception:
                continue
        
        if len(crypto_returns) > 0:
            m = calc_metrics(pd.Series(crypto_returns))
            if m:
                m['Strategy'] = strat_name
                m['Direction'] = direction
                m['Timeframe'] = tf_label
                m['Hold'] = hold_label
                m['Asset Class'] = 'Crypto'
                all_results.append(m)
        
        # --- Aggregate across GOLD symbols ---
        gold_returns = []
        for sym in GOLD_SYMBOLS:
            df = sym_data.get(sym)
            if df is None:
                continue
            try:
                signals = fn(df)
                if signals.sum() > 0:
                    rets = compute_trade_returns(df, signals, direction, hold_candles)
                    gold_returns.extend(rets.tolist())
            except Exception:
                continue
        
        if len(gold_returns) > 0:
            m = calc_metrics(pd.Series(gold_returns))
            if m:
                m['Strategy'] = strat_name
                m['Direction'] = direction
                m['Timeframe'] = tf_label
                m['Hold'] = hold_label
                m['Asset Class'] = 'Gold (XAUT+PAXG)'
                all_results.append(m)

df_all = pd.DataFrame(all_results)
print(f'\nBacktesting complete: {len(df_all)} total results.')


Backtesting on 5-Min Candle (1h hold)...



Backtesting on 15-Min Candle (3h hold)...

Backtesting on 1-Hour Candle (6h hold)...

Backtesting on 4-Hour Candle (24h hold)...

Backtesting complete: 127 total results.


## 4. Does Candle Timeframe Influence Profitability?

### 4a. Average Win Rate by Timeframe

In [21]:
# Aggregate stats by timeframe for crypto only
df_crypto = df_all[df_all['Asset Class'] == 'Crypto'].copy()

tf_summary = df_crypto.groupby('Timeframe').agg(
    Avg_Win_Rate=('Win Rate (%)', 'mean'),
    Avg_Return=('Avg Return (%)', 'mean'),
    Avg_Sharpe=('Sharpe Ratio', 'mean'),
    Avg_Profit_Factor=('Profit Factor', 'mean'),
    Total_Signals=('Signals', 'sum'),
    Strategies_Profitable=('Avg Return (%)', lambda x: (x > 0).sum()),
    Total_Strategies=('Avg Return (%)', 'count'),
).round(4)

# Reorder by candle size
tf_order = ['5-Min Candle', '15-Min Candle', '1-Hour Candle', '4-Hour Candle']
tf_summary = tf_summary.reindex(tf_order)

print('TIMEFRAME IMPACT ON STRATEGY PERFORMANCE (Crypto Pairs Only)')
print('='*90)
tf_summary

TIMEFRAME IMPACT ON STRATEGY PERFORMANCE (Crypto Pairs Only)


,Avg_Win_Rate,Avg_Return,Avg_Sharpe,Avg_Profit_Factor,Total_Signals,Strategies_Profitable,Total_Strategies
Timeframe,,,,,,,
5-Min Candle,51.7450,0.0376,1.0959,1.3403,1941,10,16
15-Min Candle,52.3119,0.0837,1.7687,1.6212,1864,11,16
1-Hour Candle,53.9106,0.1115,1.2141,1.3970,1851,11,16
4-Hour Candle,48.4331,-0.2299,-1.1705,0.8859,1911,5,16


In [22]:
# Visual: Grouped bar chart of key metrics by timeframe
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=('Avg Win Rate (%)', 'Avg Return / Trade (%)', 'Avg Sharpe Ratio', 'Avg Profit Factor'),
    horizontal_spacing=0.08
)

tf_labels = tf_summary.index.tolist()
colors = ['#FF6B6B', '#FFA94D', '#51CF66', '#339AF0']

fig.add_trace(go.Bar(x=tf_labels, y=tf_summary['Avg_Win_Rate'].tolist(), marker_color=colors, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=tf_labels, y=tf_summary['Avg_Return'].tolist(), marker_color=colors, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=tf_labels, y=tf_summary['Avg_Sharpe'].tolist(), marker_color=colors, showlegend=False), row=1, col=3)
fig.add_trace(go.Bar(x=tf_labels, y=tf_summary['Avg_Profit_Factor'].tolist(), marker_color=colors, showlegend=False), row=1, col=4)

fig.update_layout(
    height=400, template='plotly_dark',
    title_text='Does Candle Timeframe Affect Strategy Performance? (Crypto Pairs)',
)
fig.update_xaxes(tickangle=30, tickfont_size=9)
fig.show()

### 4b. Win Rate Heatmap — Strategy × Timeframe

In [23]:
pivot_wr = df_crypto.pivot_table(index='Strategy', columns='Timeframe', values='Win Rate (%)', aggfunc='first')
pivot_wr = pivot_wr.reindex(columns=tf_order)

# Sort strategies by average win rate across timeframes
pivot_wr['avg'] = pivot_wr.mean(axis=1)
pivot_wr = pivot_wr.sort_values('avg', ascending=False).drop(columns='avg')

fig = go.Figure(data=go.Heatmap(
    z=pivot_wr.values,
    x=pivot_wr.columns.tolist(),
    y=pivot_wr.index.tolist(),
    colorscale='RdYlGn',
    text=np.round(pivot_wr.values, 1),
    texttemplate='%{text}%',
    textfont={'size': 11},
    colorbar_title='Win Rate (%)'
))
fig.update_layout(
    title='Win Rate Heatmap: Strategy × Candle Timeframe (Crypto)',
    height=550, template='plotly_dark',
    yaxis=dict(autorange='reversed')
)
fig.show()

In [24]:
# Sharpe Ratio heatmap
pivot_sh = df_crypto.pivot_table(index='Strategy', columns='Timeframe', values='Sharpe Ratio', aggfunc='first')
pivot_sh = pivot_sh.reindex(columns=tf_order)
pivot_sh['avg'] = pivot_sh.mean(axis=1)
pivot_sh = pivot_sh.sort_values('avg', ascending=False).drop(columns='avg')

fig = go.Figure(data=go.Heatmap(
    z=pivot_sh.values,
    x=pivot_sh.columns.tolist(),
    y=pivot_sh.index.tolist(),
    colorscale='RdYlGn',
    text=np.round(pivot_sh.values, 2),
    texttemplate='%{text}',
    textfont={'size': 11},
    colorbar_title='Sharpe Ratio'
))
fig.update_layout(
    title='Sharpe Ratio Heatmap: Strategy × Candle Timeframe (Crypto)',
    height=550, template='plotly_dark',
    yaxis=dict(autorange='reversed')
)
fig.show()

## 5. Crypto vs Gold — Do TA Strategies Work on Gold?

Gold is a fundamentally different asset — lower volatility, macro-driven, less retail speculation. Let's see if the same strategies hold up.

In [25]:
# Compare Crypto vs Gold across all timeframes
df_gold = df_all[df_all['Asset Class'] == 'Gold (XAUT+PAXG)'].copy()

if len(df_gold) > 0:
    comparison = []
    
    for tf in tf_order:
        crypto_tf = df_crypto[df_crypto['Timeframe'] == tf]
        gold_tf = df_gold[df_gold['Timeframe'] == tf]
        
        if len(crypto_tf) > 0 and len(gold_tf) > 0:
            comparison.append({
                'Timeframe': tf,
                'Crypto Avg Win Rate (%)': round(crypto_tf['Win Rate (%)'].mean(), 2),
                'Gold Avg Win Rate (%)': round(gold_tf['Win Rate (%)'].mean(), 2),
                'Crypto Avg Return (%)': round(crypto_tf['Avg Return (%)'].mean(), 4),
                'Gold Avg Return (%)': round(gold_tf['Avg Return (%)'].mean(), 4),
                'Crypto Avg Sharpe': round(crypto_tf['Sharpe Ratio'].mean(), 4),
                'Gold Avg Sharpe': round(gold_tf['Sharpe Ratio'].mean(), 4),
                'Crypto Total Signals': int(crypto_tf['Signals'].sum()),
                'Gold Total Signals': int(gold_tf['Signals'].sum()),
            })
    
    df_compare = pd.DataFrame(comparison)
    print('CRYPTO vs GOLD — Strategy Performance Comparison')
    print('='*100)
    display(df_compare)
else:
    print('No Gold strategy results available (insufficient signals).')

CRYPTO vs GOLD — Strategy Performance Comparison


,Timeframe,Crypto Avg Win Rate (%),Gold Avg Win Rate (%),Crypto Avg Return (%),Gold Avg Return (%),Crypto Avg Sharpe,Gold Avg Sharpe,Crypto Total Signals,Gold Total Signals
0,5-Min Candle,51.74,54.70,0.0376,0.0324,1.0959,2.6094,1941,473
1,15-Min Candle,52.31,54.28,0.0837,0.0511,1.7687,2.2346,1864,517
2,1-Hour Candle,53.91,42.52,0.1115,-0.0278,1.2141,-1.1792,1851,478
3,4-Hour Candle,48.43,49.03,-0.2299,-0.0694,-1.1705,-0.2423,1911,579


In [26]:
# Side-by-side bar chart: Crypto vs Gold
if len(df_gold) > 0 and len(df_compare) > 0:
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=('Avg Win Rate (%)', 'Avg Return / Trade (%)', 'Avg Sharpe Ratio'),
        horizontal_spacing=0.08
    )
    
    tfs = df_compare['Timeframe'].tolist()
    
    fig.add_trace(go.Bar(x=tfs, y=df_compare['Crypto Avg Win Rate (%)'].tolist(), name='Crypto', marker_color='#51CF66'), row=1, col=1)
    fig.add_trace(go.Bar(x=tfs, y=df_compare['Gold Avg Win Rate (%)'].tolist(), name='Gold', marker_color='#FFD43B'), row=1, col=1)
    
    fig.add_trace(go.Bar(x=tfs, y=df_compare['Crypto Avg Return (%)'].tolist(), name='Crypto', marker_color='#51CF66', showlegend=False), row=1, col=2)
    fig.add_trace(go.Bar(x=tfs, y=df_compare['Gold Avg Return (%)'].tolist(), name='Gold', marker_color='#FFD43B', showlegend=False), row=1, col=2)
    
    fig.add_trace(go.Bar(x=tfs, y=df_compare['Crypto Avg Sharpe'].tolist(), name='Crypto', marker_color='#51CF66', showlegend=False), row=1, col=3)
    fig.add_trace(go.Bar(x=tfs, y=df_compare['Gold Avg Sharpe'].tolist(), name='Gold', marker_color='#FFD43B', showlegend=False), row=1, col=3)
    
    fig.update_layout(height=400, template='plotly_dark', title_text='Crypto vs Gold: Do TA Strategies Transfer?', barmode='group')
    fig.update_xaxes(tickangle=30, tickfont_size=9)
    fig.show()
else:
    print('Insufficient Gold data for visualization.')

### 5b. Per-Strategy Crypto vs Gold Breakdown (1-Hour Candle)

In [27]:
# Detailed per-strategy comparison on 1h timeframe
tf_focus = '1-Hour Candle'
crypto_1h = df_crypto[df_crypto['Timeframe'] == tf_focus].set_index('Strategy')
gold_1h = df_gold[df_gold['Timeframe'] == tf_focus].set_index('Strategy') if len(df_gold) > 0 else pd.DataFrame()

if not gold_1h.empty:
    merged = crypto_1h[['Win Rate (%)', 'Avg Return (%)', 'Sharpe Ratio', 'Signals']].rename(
        columns=lambda c: f'Crypto {c}'
    ).join(
        gold_1h[['Win Rate (%)', 'Avg Return (%)', 'Sharpe Ratio', 'Signals']].rename(
            columns=lambda c: f'Gold {c}'
        ), how='outer'
    )
    merged = merged.fillna('-')
    print(f'PER-STRATEGY COMPARISON: CRYPTO vs GOLD ({tf_focus})')
    print('='*110)
    display(merged)
else:
    print('No Gold results for 1-hour timeframe.')

PER-STRATEGY COMPARISON: CRYPTO vs GOLD (1-Hour Candle)


,Crypto Win Rate (%),Crypto Avg Return (%),Crypto Sharpe Ratio,Crypto Signals,Gold Win Rate (%),Gold Avg Return (%),Gold Sharpe Ratio,Gold Signals
Strategy,,,,,,,,
BB Lower Bounce,61.29,0.2546,2.9677,93,38.46,-0.0305,-1.1481,26
BB Squeeze Breakout,52.00,-0.1243,-2.8636,75,20.00,0.2015,5.4934,5
BB Upper Reject,52.38,0.0654,0.9466,126,71.43,0.0135,0.5343,21
Bearish MACD Cross,46.75,-0.0558,-0.6794,154,69.23,0.0733,2.2223,39
Bullish MACD Cross,57.23,0.1108,1.6679,159,33.33,-0.0772,-1.7500,39
EMA 200/50 Pullback,54.39,0.0791,1.1792,114,36.36,0.0466,1.6064,11
EMA 9/21 Death Cross,43.82,-0.0462,-0.6198,89,56.00,0.1147,5.0469,25
EMA 9/21 Golden Cross,50.00,0.1057,1.4174,96,32.00,-0.0518,-1.3197,25
Mean Reversion,55.66,0.2026,2.2965,106,41.67,-0.0913,-4.8350,48


## 6. Best Strategy per Timeframe — Grand Summary

In [28]:
# Find the best performing strategy for each timeframe
print('BEST STRATEGY PER TIMEFRAME (Crypto — Ranked by Sharpe Ratio)')
print('='*110)

best_per_tf = []
for tf in tf_order:
    subset = df_crypto[df_crypto['Timeframe'] == tf].copy()
    if len(subset) == 0:
        continue
    best = subset.sort_values('Sharpe Ratio', ascending=False).iloc[0]
    best_per_tf.append({
        'Timeframe': tf,
        'Best Strategy': best['Strategy'],
        'Direction': best['Direction'],
        'Signals': int(best['Signals']),
        'Win Rate (%)': best['Win Rate (%)'],
        'Avg Return (%)': best['Avg Return (%)'],
        'Total Return (%)': best['Total Return (%)'],
        'Profit Factor': best['Profit Factor'],
        'Sharpe Ratio': best['Sharpe Ratio'],
    })

df_best = pd.DataFrame(best_per_tf)
display(df_best)

BEST STRATEGY PER TIMEFRAME (Crypto — Ranked by Sharpe Ratio)


,Timeframe,Best Strategy,Direction,Signals,Win Rate (%),Avg Return (%),Total Return (%),Profit Factor,Sharpe Ratio
0,5-Min Candle,BB Lower Bounce,LONG,103,56.31,0.1495,16.4622,2.0676,4.3949
1,15-Min Candle,Volume Surge + RSI Dip,LONG,136,61.76,0.3318,56.1463,3.1820,6.1631
2,1-Hour Candle,Volume Surge + RSI Dip,LONG,111,63.06,0.5940,91.2402,3.4497,7.2966
3,4-Hour Candle,BB Squeeze Breakout,LONG,41,51.22,0.6770,30.0504,1.8937,4.0537


In [29]:
# Full ranked table for the best-performing timeframe
best_tf = df_best.sort_values('Sharpe Ratio', ascending=False).iloc[0]['Timeframe']
df_best_tf = df_crypto[df_crypto['Timeframe'] == best_tf].copy()

# Composite score
if len(df_best_tf) > 1:
    for col in ['Win Rate (%)', 'Profit Factor', 'Sharpe Ratio']:
        cmin, cmax = df_best_tf[col].min(), df_best_tf[col].max()
        rng = cmax - cmin if cmax != cmin else 1
        df_best_tf[f'{col}_n'] = (df_best_tf[col] - cmin) / rng
    df_best_tf['Composite'] = (
        df_best_tf['Win Rate (%)_n'] * 0.30 +
        df_best_tf['Profit Factor_n'] * 0.35 +
        df_best_tf['Sharpe Ratio_n'] * 0.35
    ).round(4)
    df_best_tf = df_best_tf.sort_values('Composite', ascending=False)

print(f'\nFULL RANKING ON BEST TIMEFRAME: {best_tf}')
print('='*110)

show_cols = ['Strategy', 'Direction', 'Signals', 'Win Rate (%)', 'Avg Return (%)',
             'Total Return (%)', 'Max Drawdown (%)', 'Profit Factor', 'Sharpe Ratio']
if 'Composite' in df_best_tf.columns:
    show_cols.append('Composite')

df_show = df_best_tf[show_cols].reset_index(drop=True)
df_show.index = df_show.index + 1
df_show.index.name = 'Rank'
display(df_show)

winner = df_show.iloc[0]
print(f'\n{"="*70}')
print(f'  OVERALL BEST: {winner["Strategy"]} on {best_tf}')
print(f'  Direction: {winner["Direction"]}  |  Win Rate: {winner["Win Rate (%)"]:.1f}%  |  Avg Return: {winner["Avg Return (%)"]:.4f}%')
print(f'  Profit Factor: {winner["Profit Factor"]:.2f}  |  Sharpe: {winner["Sharpe Ratio"]:.2f}')
print(f'{"="*70}')


FULL RANKING ON BEST TIMEFRAME: 1-Hour Candle


,Strategy,Direction,Signals,Win Rate (%),Avg Return (%),Total Return (%),Max Drawdown (%),Profit Factor,Sharpe Ratio,Composite
Rank,,,,,,,,,,
1,Volume Surge + RSI Dip,LONG,111,63.06,0.5940,91.2402,-7.3976,3.4497,7.2966,1.0000
2,StochRSI Oversold,LONG,179,62.01,0.3915,97.7105,-3.9348,2.6730,4.3032,0.7844
3,BB Lower Bounce,LONG,93,61.29,0.2546,25.6034,-9.7674,1.6325,2.9677,0.5984
4,RSI Oversold Bounce,LONG,101,56.44,0.2501,27.1910,-8.8198,1.7820,2.5373,0.5264
5,Mean Reversion,LONG,106,55.66,0.2026,22.6649,-14.9791,1.4722,2.2965,0.4676
6,Bullish MACD Cross,LONG,159,57.23,0.1108,18.2153,-6.2335,1.3290,1.6679,0.4527
7,EMA 200/50 Pullback,LONG,114,54.39,0.0791,8.7380,-9.4857,1.2211,1.1792,0.3783
8,StochRSI Overbought,SHORT,194,53.61,0.1066,20.7160,-13.7236,1.2473,1.2282,0.3710
9,Triple EMA Stack,LONG,77,55.84,0.0224,1.2257,-10.0992,1.0541,0.3070,0.3502



  OVERALL BEST: Volume Surge + RSI Dip on 1-Hour Candle
  Direction: LONG  |  Win Rate: 63.1%  |  Avg Return: 0.5940%
  Profit Factor: 3.45  |  Sharpe: 7.30


## 7. Key Findings & Conclusions

In [30]:
# Auto-generate insights
print('KEY FINDINGS')
print('='*70)

# 1. Which timeframe generates most signals?
signal_by_tf = df_crypto.groupby('Timeframe')['Signals'].sum()
most_signals_tf = signal_by_tf.idxmax()
print(f'\n1. SIGNAL FREQUENCY:')
for tf in tf_order:
    if tf in signal_by_tf.index:
        print(f'   {tf}: {int(signal_by_tf[tf])} total signals')

# 2. Which timeframe has highest average Sharpe?
sharpe_by_tf = df_crypto.groupby('Timeframe')['Sharpe Ratio'].mean()
best_sharpe_tf = sharpe_by_tf.reindex(tf_order).idxmax()
print(f'\n2. BEST RISK-ADJUSTED TIMEFRAME: {best_sharpe_tf}')
print(f'   Average Sharpe Ratio: {sharpe_by_tf[best_sharpe_tf]:.4f}')

# 3. Crypto vs Gold verdict
if len(df_gold) > 0:
    crypto_avg_sharpe = df_crypto['Sharpe Ratio'].mean()
    gold_avg_sharpe = df_gold['Sharpe Ratio'].mean()
    crypto_avg_wr = df_crypto['Win Rate (%)'].mean()
    gold_avg_wr = df_gold['Win Rate (%)'].mean()
    print(f'\n3. CRYPTO vs GOLD:')
    print(f'   Crypto — Avg Win Rate: {crypto_avg_wr:.2f}%, Avg Sharpe: {crypto_avg_sharpe:.4f}')
    print(f'   Gold   — Avg Win Rate: {gold_avg_wr:.2f}%, Avg Sharpe: {gold_avg_sharpe:.4f}')
    if gold_avg_sharpe > crypto_avg_sharpe:
        print(f'   Verdict: Gold shows BETTER risk-adjusted returns with TA strategies')
    else:
        print(f'   Verdict: Crypto shows BETTER risk-adjusted returns with TA strategies')
    
    gold_profitable = df_gold[df_gold['Avg Return (%)'] > 0]
    print(f'   Gold profitable strategies: {len(gold_profitable)} / {len(df_gold)}')

# 4. Does shorter timeframe = more noise?
print(f'\n4. TIMEFRAME vs NOISE:')
for tf in tf_order:
    subset = df_crypto[df_crypto['Timeframe'] == tf]
    if len(subset) > 0:
        profitable = (subset['Avg Return (%)'] > 0).sum()
        total = len(subset)
        print(f'   {tf}: {profitable}/{total} strategies profitable ({profitable/total*100:.0f}%)')

print(f'\n{"="*70}')

KEY FINDINGS

1. SIGNAL FREQUENCY:
   5-Min Candle: 1941 total signals
   15-Min Candle: 1864 total signals
   1-Hour Candle: 1851 total signals
   4-Hour Candle: 1911 total signals

2. BEST RISK-ADJUSTED TIMEFRAME: 15-Min Candle
   Average Sharpe Ratio: 1.7687

3. CRYPTO vs GOLD:
   Crypto — Avg Win Rate: 51.60%, Avg Sharpe: 0.7270
   Gold   — Avg Win Rate: 50.06%, Avg Sharpe: 0.8278
   Verdict: Gold shows BETTER risk-adjusted returns with TA strategies
   Gold profitable strategies: 36 / 63

4. TIMEFRAME vs NOISE:
   5-Min Candle: 10/16 strategies profitable (62%)
   15-Min Candle: 11/16 strategies profitable (69%)
   1-Hour Candle: 11/16 strategies profitable (69%)
   4-Hour Candle: 5/16 strategies profitable (31%)

